Links scraping

In [ ]:
from datetime import datetime, timedelta
import json
import time
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

BASE = "https://www.bbc.com"
WAYBACK = "https://web.archive.org/web"

def normalize_url(url: str) -> str:
    return url.split("#")[0].split("?")[0]

def collect_article_links(date: str, category: str) -> list[str]:
    # Use Wayback Machine ONLY to find what links existed on that date
    entry_url = f"{WAYBACK}/{date}000000/{BASE}/{category}"

    headers = {"User-Agent": "SchoolAssignmentBot/1.0"}
    try:
        r = requests.get(entry_url, headers=headers, timeout=30)
        r.raise_for_status()
    except Exception as e:
        print(f"Could not fetch archive for {date}: {e}")
        return []

    soup = BeautifulSoup(r.text, "lxml")

    links: set[str] = set()
    for a in soup.find_all("a", href=True):
        href = a["href"]
        full = urljoin(entry_url, href)

        # Extract the original BBC link from the archive wrapper
        if "bbc.com" in full and "/articles/" in full:
            if "https://www.bbc.com" in full:
                clean_url = "https://www.bbc.com" + full.split("https://www.bbc.com")[-1]
                links.add(normalize_url(clean_url))

    return sorted(links)

def scrape_all_links(category: str, max_days_back: int, max_links: int, wait_s: int):
    all_links = set()

    # Start with today's date
    current_date_obj = datetime.now()

    print(f"Starting link collection for category: {category}")

    # Loop until we have enough links or we've gone back too far
    days_back = 0
    while len(all_links) < max_links and days_back < max_days_back:
        date_str = current_date_obj.strftime("%Y%m%d")
        print(f"Searching archive for: {date_str}...")

        new_links = collect_article_links(date_str, category)
        all_links.update(new_links)

        print(f"Found {len(new_links)} links. Total unique links: {len(all_links)}")

        # Move to the previous day
        current_date_obj -= timedelta(days=1)
        days_back += 1
        time.sleep(wait_s) # Small delay to be polite to the archive

    links_list = sorted(list(all_links))
    
    # Save links to a file
    links_file = f"all_{category}_collected_links.json"
    with open(links_file, "w", encoding="utf-8") as f:
        json.dump(links_list, f, indent=2)
        
    print(f"\nSaved {len(links_list)} unique links to {links_file}")

if __name__ == "__main__":
  scrape_all_links(category="technology", max_days_back=60, max_links=1000, wait_s=15)

Article scraping

In [ ]:
import json
import time
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse

BASE = "https://www.bbc.com"
WAYBACK = "https://web.archive.org/web"

def category_from_url(url: str) -> str | None:
    path = urlparse(url).path.strip("/")
    if not path:
        return None
    return path.split("/")[0]

def scrape_article(url: str, category_hint: str | None = None) -> dict:
    # Now fetching DIRECTLY from BBC, not via Wayback
    headers = {"User-Agent": "SchoolAssignmentBot/1.0"}
    r = requests.get(url, headers=headers, timeout=30)
    r.raise_for_status()

    soup = BeautifulSoup(r.text, "lxml")

    # Title
    h1 = soup.find("h1")
    title = h1.get_text(" ", strip=True) if h1 else None
    if not title and soup.title:
        title = soup.title.get_text(" ", strip=True)

    # Content
    main = soup.find("main")
    scope = main if main else soup
    paragraphs = [p.get_text(" ", strip=True) for p in scope.find_all("p") if len(p.get_text()) > 40]

    cleaned = []
    for t in paragraphs:
        if not cleaned or cleaned[-1] != t:
            cleaned.append(t)

    content = "\n\n".join(cleaned) if cleaned else None

    return {
        "url": url,
        "category": category_hint or category_from_url(url),
        "title": title,
        "content": content,
    }

def build_dataset(input_file: str, category: str, max_articles: int = 30, delay_s: float = 1.0) -> list[dict]:
    # Step 1: Get historical links from Wayback
    with open(input_file, "r", encoding="utf-8") as f:
        urls = json.load(f)
    
    urls = urls[:max_articles]
    print(f"Loaded {len(urls)} links from {input_file}. Starting scrape...")

    dataset = []
    for i, url in enumerate(urls, start=1):
        try:
            # Step 2: Scrape the live version of those links
            item = scrape_article(url, category_hint=category)
            dataset.append(item)
            print(f"[{i}/{len(urls)}] OK - {item['title']}")
        except Exception as e:
            dataset.append({"url": url, "error": str(e)})
            print(f"[{i}/{len(urls)}] Error - {url}")
        time.sleep(delay_s)

    return dataset

if __name__ == "__main__":
    # Historical date to browse, but scraping will be live
    category = "sport"
    filename = "724_all_sport_collected_links.json"
    # Increased max_articles from 5 to 30
    data = build_dataset(filename, category, max_articles=800, delay_s=5)

    out_file = f"bbc_{category}_articles.json"
    with open(out_file, "w", encoding="utf-8") as f:
        json.dump(data, f, ensure_ascii=False, indent=2)

    print(f"\nSaved {len(data)} items (live content) to {out_file}")